In [2]:
import pandas as pd
import sqlite3

# Loading cleaned datasets
providers = pd.read_csv("Providers_Data_Cleaned.csv")
receivers = pd.read_csv("Receivers_Data_Cleaned.csv")
food = pd.read_csv("Food_Listings_Data_Cleaned.csv")
claims = pd.read_csv("Claims_Data_Cleaned.csv")

# Create SQLite database
conn = sqlite3.connect("food_waste_management.db")

# Create tables
providers.to_sql("providers", conn, if_exists="replace", index=False)
receivers.to_sql("receivers", conn, if_exists="replace", index=False)
food.to_sql("food_listings", conn, if_exists="replace", index=False)
claims.to_sql("claims", conn, if_exists="replace", index=False)

print("Tables loaded successfully!")

Tables loaded successfully!


In [13]:
#How many food providers and receivers are there in each city?

query = """
SELECT
    p.City,
    p.Provider_Count,
    COALESCE(r.Receiver_Count, 0) AS Receiver_Count
FROM
(
    SELECT City, COUNT(*) AS Provider_Count
    FROM providers
    GROUP BY City
) p
LEFT JOIN
(
    SELECT City, COUNT(*) AS Receiver_Count
    FROM receivers
    GROUP BY City
) r
ON p.City = r.City
ORDER BY p.Provider_Count DESC;
"""

result = pd.read_sql_query(query, conn)
print(result.head(1000))

                         City  Provider_Count  Receiver_Count
0                   New Carol               3               0
1    South Christopherborough               3               0
2                 Bradleyport               2               0
3                  Davidville               2               0
4                East Anthony               2               0
..                        ...             ...             ...
958               Wrightville               1               0
959                 Yatesside               1               0
960              Youngchester               1               0
961              Zimmermanton               1               0
962            Zimmermanville               1               0

[963 rows x 3 columns]


In [14]:
#Which type of food provider contributes the most food?

query = """
SELECT
    p.Type,
    SUM(f.Quantity) AS Total_Quantity
FROM providers p
JOIN food_listings f
ON p.Provider_ID = f.Provider_ID
GROUP BY p.Type
ORDER BY Total_Quantity DESC;
"""

print(pd.read_sql_query(query, conn))

               Type  Total_Quantity
0        Restaurant            6923
1       Supermarket            6696
2  Catering Service            6116
3     Grocery Store            6059


In [18]:
#Contact information of food providers in a specific city

query = """
SELECT
    Name,
    Contact,
    Address
FROM providers
WHERE City = 'New Carol';
"""

result = pd.read_sql_query(query, conn)
print(result.head())

                Name      Contact  \
0  Bradford-Martinez   1994510254   
1        Hammond LLC  13244824894   
2       Hill-Russell    756309218   

                                           Address  
0  366 Wheeler Fields\nHarringtonchester, WY 86540  
1             7396 Patrick Row\nEast Roy, UT 18983  
2    68533 Martin Fork\nSouth Morganfort, IL 17612  


In [20]:
#Which receivers have claimed the most food?

query = """
SELECT
    r.Name,
    COUNT(c.Claim_ID) AS Total_Claims
FROM receivers r
JOIN claims c
ON r.Receiver_ID = c.Receiver_ID
GROUP BY r.Receiver_ID, r.Name
ORDER BY Total_Claims DESC;
"""

result = pd.read_sql_query(query, conn)
print(result.head())

                Name  Total_Claims
0       Scott Hunter             5
1  William Frederick             5
2       Matthew Webb             5
3     Anthony Garcia             5
4         Alvin West             4


In [21]:
#Total quantity of food available from all providers

query = """
SELECT
    SUM(Quantity) AS Total_Food_Available
FROM food_listings;
"""

result = pd.read_sql_query(query, conn)
print(result.head())

   Total_Food_Available
0                 25794


In [22]:
#Which city has the highest number of food listings?

query = """
SELECT
    p.City,
    COUNT(f.Food_ID) AS Listing_Count
FROM providers p
JOIN food_listings f
ON p.Provider_ID = f.Provider_ID
GROUP BY p.City
ORDER BY Listing_Count DESC
LIMIT 1;
"""

result = pd.read_sql_query(query, conn)
print(result.head())

            City  Listing_Count
0  South Kathryn              6


In [23]:
#Most commonly available food types

query = """
SELECT
    Food_Type,
    COUNT(*) AS Frequency
FROM food_listings
GROUP BY Food_Type
ORDER BY Frequency DESC;
"""

result = pd.read_sql_query(query, conn)
print(result.head())

        Food_Type  Frequency
0      Vegetarian        336
1           Vegan        334
2  Non-Vegetarian        330


In [24]:
#How many food claims have been made for each food item?

query = """
SELECT
    f.Food_Name,
    COUNT(c.Claim_ID) AS Total_Claims
FROM food_listings f
LEFT JOIN claims c
ON f.Food_ID = c.Food_ID
GROUP BY f.Food_ID, f.Food_Name
ORDER BY Total_Claims DESC;
"""

result = pd.read_sql_query(query, conn)
print(result.head())

  Food_Name  Total_Claims
0      Soup             5
1   Chicken             5
2      Fish             5
3      Rice             4
4   Chicken             4


In [25]:
#Which provider has had the highest number of successful food claims?

query = """
SELECT
    p.Name,
    COUNT(c.Claim_ID) AS Successful_Claims
FROM providers p
JOIN food_listings f
ON p.Provider_ID = f.Provider_ID
JOIN claims c
ON f.Food_ID = c.Food_ID
WHERE c.Status = 'Completed'
GROUP BY p.Provider_ID, p.Name
ORDER BY Successful_Claims DESC
LIMIT 1;
"""

result = pd.read_sql_query(query, conn)
print(result.head())

          Name  Successful_Claims
0  Barry Group                  5


In [26]:
#Percentage of completed vs pending vs canceled claims

query = """
SELECT
    Status,
    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*) FROM claims),
        2
    ) AS Percentage
FROM claims
GROUP BY Status;
"""

result = pd.read_sql_query(query, conn)
print(result.head())

      Status  Percentage
0  Cancelled        33.6
1  Completed        33.9
2    Pending        32.5


In [29]:
#Average quantity of food claimed per receiver

query = """
SELECT
    r.Name,
    ROUND(AVG(f.Quantity), 2) AS Avg_Quantity_Claimed
FROM receivers r
JOIN claims c
ON r.Receiver_ID = c.Receiver_ID
JOIN food_listings f
ON c.Food_ID = f.Food_ID
GROUP BY r.Receiver_ID, r.Name
ORDER BY Avg_Quantity_Claimed DESC;
"""

result = pd.read_sql_query(query, conn)
print(result.head())

                 Name  Avg_Quantity_Claimed
0         Nancy Silva                  50.0
1          Lisa Pitts                  50.0
2     Daniel Williams                  50.0
3        Peggy Knight                  50.0
4  Christopher Wright                  50.0


In [30]:
query = """
SELECT
    ROUND(AVG(Quantity), 2) AS Avg_Quantity
FROM food_listings f
JOIN claims c
ON f.Food_ID = c.Food_ID;
"""

result = pd.read_sql_query(query, conn)
print(result.head())

   Avg_Quantity
0         25.96


In [31]:
#Which meal type is claimed the most?
query = """
SELECT
    f.Meal_Type,
    COUNT(c.Claim_ID) AS Total_Claims
FROM food_listings f
JOIN claims c
ON f.Food_ID = c.Food_ID
GROUP BY f.Meal_Type
ORDER BY Total_Claims DESC;
"""

result = pd.read_sql_query(query, conn)
print(result.head())

   Meal_Type  Total_Claims
0  Breakfast           278
1      Lunch           250
2     Snacks           240
3     Dinner           232


In [32]:
#Total quantity of food donated by each provider

query = """
SELECT
    p.Name,
    SUM(f.Quantity) AS Total_Donated
FROM providers p
JOIN food_listings f
ON p.Provider_ID = f.Provider_ID
GROUP BY p.Provider_ID, p.Name
ORDER BY Total_Donated DESC;
"""

result = pd.read_sql_query(query, conn)
print(result.head())


                         Name  Total_Donated
0                 Barry Group            179
1  Evans, Wright and Mitchell            158
2                 Smith Group            150
3                  Nelson LLC            142
4                  Ruiz-Oneal            140
